# 🏟️ 对抗学习量化系统 - 庄散对抗竞技场 (云端版)

**一键运行**: 点击菜单 → 代码执行程序 → 全部运行 (Ctrl+F9)

**预计时间**: 
- 快速模式 (2组合×100轮): ~15分钟 (GPU) / ~30分钟 (CPU)
- 标准模式 (8组合×500轮): ~4-6小时 (GPU) / ~8-12小时 (CPU)

**流程**:
1. 安装依赖
2. 下载脚本
3. 生成假数据
4. 运行竞技场
5. 查看结果


In [ ]:
# ========================================
# Step 1: 安装依赖
# ========================================
import subprocess, sys

print("★ 安装依赖...")
subprocess.run([sys.executable, "-m", "pip", "install", 
                "torch", "numpy", "scipy", "pandas", "--quiet"], check=True)

import torch
print(f"✓ torch {torch.__version__}")
print(f"✓ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU模式'}")
import numpy as np
print(f"✓ numpy {np.__version__}")
print("✓ 依赖安装完成!")

In [ ]:
# ========================================
# Step 2: 下载脚本 & 创建目录
# ========================================
import os, urllib.request
from pathlib import Path

BASE = Path("/content/AdversarialLearning")
DOTPY = BASE / "dotpy"
ADV_DATA = BASE / "adversarial data"
ADV_MODEL = BASE / "adversarial model"
RESULTS = BASE / "stockresults"
STOCKDATA = BASE / "stockdata"

for d in [DOTPY, ADV_DATA, ADV_MODEL, RESULTS, STOCKDATA]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ 创建目录: {d}")

# 从GitHub下载脚本
GITHUB = "https://raw.githubusercontent.com/Scilogos/scilogos.github.io/main/cai/dotpy"
scripts = [
    "stock_config.py",
    "adversarial_env.py", 
    "market_generator.py",
    "stock_data_manager.py",
    "stock_interpreter.py",
    "run_pipeline.py",
    "feedback_processor.py",
    "pipeline_monitor.py",
]

print(f"\n★ 下载脚本 (从GitHub)...")
for s in scripts:
    url = f"{GITHUB}/{s}"
    dest = DOTPY / s
    try:
        urllib.request.urlretrieve(url, str(dest))
        size = dest.stat().st_size / 1024
        print(f"  ✓ {s}: {size:.1f}KB")
    except Exception as e:
        print(f"  ✗ {s}: {e}")

# ★ 修改stock_config.py的路径为Linux版本
config_path = DOTPY / "stock_config.py"
content = config_path.read_text(encoding="utf-8")
# 替换Windows路径为Linux路径
content = content.replace(
    'BASE_DIR = Path(r"C:\\Users\\HUAWEI\\Desktop\\Adversarial Learning")',
    f'BASE_DIR = Path("{BASE}")'
)
content = content.replace(
    'BASE_DIR = Path(r"C:\Users\HUAWEI\Desktop\Adversarial Learning")',
    f'BASE_DIR = Path("{BASE}")'
)
# 也处理单引号版本
import re
content = re.sub(
    r'BASE_DIR\s*=\s*Path\(r?".*?Adversarial Learning"?\)',
    f'BASE_DIR = Path("{BASE}")',
    content
)
config_path.write_text(content, encoding="utf-8")
print(f"\n✓ stock_config.py 路径已修改为: {BASE}")

# 替换Python解释器路径
content = config_path.read_text(encoding="utf-8")
content = re.sub(
    r'PYTHON_EXE\s*=\s*r?".*?python.*?"',
    'PYTHON_EXE = "python3"',
    content,
    flags=re.IGNORECASE
)
config_path.write_text(content, encoding="utf-8")
print("✓ Python解释器路径已修改为: python3")
print("\n★ 准备完成!")

In [ ]:
# ========================================
# Step 3: 生成逼真A股假数据 (GBM模型)
# ========================================
import numpy as np
from pathlib import Path

BASE = Path("/content/AdversarialLearning")
ADV_DATA = BASE / "adversarial data"

def generate_fake_prices(n_samples=2000, seq_len=30, n_features=6, seed=42):
    """GBM生成逼真A股数据: (N, 30, 6) = [open, high, low, close, volume, pct]"""
    np.random.seed(seed)
    data = np.zeros((n_samples, seq_len, n_features))
    
    for i in range(n_samples):
        p0 = np.random.uniform(5, 50)
        mu = np.random.uniform(-0.001, 0.002)
        sigma = np.random.uniform(0.01, 0.04)
        
        returns = np.random.normal(mu, sigma, seq_len)
        returns = np.clip(returns, -0.10, 0.10)  # 涨跌停
        close = p0 * np.cumprod(1 + returns)
        
        iv = sigma * 0.5
        open_price = close * (1 + np.random.normal(0, iv*0.3, seq_len))
        high_price = np.maximum(open_price, close) * (1 + np.abs(np.random.normal(0, iv, seq_len)))
        low_price = np.minimum(open_price, close) * (1 - np.abs(np.random.normal(0, iv, seq_len)))
        volume = np.random.lognormal(12, 1.5, seq_len).astype(float)
        
        pct = np.zeros(seq_len)
        pct[0] = returns[0]
        pct[1:] = np.diff(close) / (close[:-1] + 1e-8)
        
        data[i] = np.stack([open_price, high_price, low_price, close, volume, pct], axis=1)
    
    return data

print("★ 生成假数据: 2000条 × 30天 × 6特征...")
fake_data = generate_fake_prices(2000)
data_path = ADV_DATA / "generated_2000.npy"
np.save(str(data_path), fake_data)

print(f"✓ 已保存: {data_path}")
print(f"  形状: {fake_data.shape}")
print(f"  价格范围: [{fake_data[:,:,3].min():.2f}, {fake_data[:,:,3].max():.2f}]")
print(f"  成交量范围: [{fake_data[:,:,4].min():.0f}, {fake_data[:,:,4].max():.0f}]")
print(f"  涨跌幅范围: [{fake_data[:,:,5].min():.4f}, {fake_data[:,:,5].max():.4f}]")
print("\n★ 数据质量检查:")
print(f"  无NaN: {not np.any(np.isnan(fake_data))}")
print(f"  无Inf: {not np.any(np.isinf(fake_data))}")
print(f"  全正价格: {(fake_data[:,:,:4] > 0).all()}")

In [ ]:
# ========================================
# Step 4: 运行庄散对抗竞技场
# ========================================
# ★ 标准模式: 8组合×500轮 (预计4-8小时)
# 如果想快速测试，修改 --arena-episodes 为 100，--arena-combos 为 2

import subprocess, sys, time
from pathlib import Path
from datetime import datetime

BASE = Path("/content/AdversarialLearning")
DOTPY = BASE / "dotpy"
DATA_PATH = BASE / "adversarial data" / "generated_2000.npy"
RESULTS = BASE / "stockresults"

print("=" * 60)
print("  ★ 庄散对抗竞技场 启动")
print("=" * 60)
print(f"  时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  数据: {DATA_PATH}")
print(f"  组合: 全部8种")
print(f"  轮数: 500/组合")
print(f"  总tick: {8 * 500 * 240:,}")
print("=" * 60)

# 运行
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(DOTPY / "adversarial_env.py"),
     "--mode", "arena",
     "--price-data", str(DATA_PATH),
     "--benchmark-source", "fake",
     "--arena-episodes", "500"],
    cwd=str(DOTPY),
    capture_output=False,
    text=True,
)
elapsed = time.time() - t0

print(f"\n★ 完成! 耗时: {elapsed/60:.1f} 分钟")
print(f"  退出码: {result.returncode}")

In [ ]:
# ========================================
# Step 5: 查看结果
# ========================================
import json
from pathlib import Path

RESULTS = Path("/content/AdversarialLearning/stockresults")

# 检查arena结果
arena_file = RESULTS / "arena_results.json"
if arena_file.exists():
    with open(arena_file, "r", encoding="utf-8") as f:
        results = json.load(f)
    
    print("=" * 60)
    print("  ★ 竞技场结果")
    print("=" * 60)
    print(json.dumps(results, indent=2, ensure_ascii=False))
else:
    print("⚠ 未找到 arena_results.json")
    print("  可能训练尚未完成或出错")
    
# 列出所有结果文件
print("\n★ 结果目录:")
for f in sorted(RESULTS.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name}: {size:.1f}KB")

In [ ]:
# ========================================
# Step 6: 下载结果到本地
# ========================================
from google.colab import files
from pathlib import Path

RESULTS = Path("/content/AdversarialLearning/stockresults")

print("★ 下载结果文件...")
for f in RESULTS.glob("*.json"):
    print(f"  下载: {f.name}")
    files.download(str(f))

print("\n✓ 全部下载完成!")

In [ ]:
# ========================================
# 附: 快速测试版 (2组合×100轮, ~15分钟)
# ========================================
# 取消下面代码的注释来运行快速测试

"""
import subprocess, sys, time
from pathlib import Path

BASE = Path("/content/AdversarialLearning")
DOTPY = BASE / "dotpy"
DATA_PATH = BASE / "adversarial data" / "generated_2000.npy"

# 先用小数据集
import numpy as np
small_data = np.load(str(DATA_PATH))[:500]  # 只取500条
small_path = BASE / "adversarial data" / "generated_500.npy"
np.save(str(small_path), small_data)

t0 = time.time()
subprocess.run(
    [sys.executable, str(DOTPY / "adversarial_env.py"),
     "--mode", "arena",
     "--price-data", str(small_path),
     "--benchmark-source", "fake",
     "--arena-combos", "2",
     "--arena-episodes", "100"],
    cwd=str(DOTPY),
)
print(f"快速测试完成: {(time.time()-t0)/60:.1f}分钟")
"""